In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from load import *
from aligner import *
from features.windowing import create_windows


[*********************100%***********************]  5 of 5 completed


# 2. Validate

In [2]:
# Missing values
data.isna().sum()

Price   Ticker
Close   GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
High    GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Low     GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Open    GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Volume  GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
dtype: int64

In [3]:
data.index.duplicated().sum()
print(data.shape)
print(data.head())
print(data.columns)

(6539, 25)
Price      Close                               High                            \
Ticker       GLD        QQQ        SPY TLT VNQ  GLD        QQQ        SPY TLT   
Date                                                                            
2000-01-03   NaN  79.839691  91.132767 NaN NaN  NaN  81.050979  92.895111 NaN   
2000-01-04   NaN  74.362518  87.568886 NaN NaN  NaN  78.786351  90.271146 NaN   
2000-01-05   NaN  72.466629  87.725533 NaN NaN  NaN  75.521182  88.685031 NaN   
2000-01-06   NaN  67.489784  86.315674 NaN NaN  NaN  74.151866  88.665465 NaN   
2000-01-07   NaN  75.837158  91.328583 NaN NaN  NaN  75.837158  91.328583 NaN   

Price           ... Open                               Volume            \
Ticker     VNQ  ...  GLD        QQQ        SPY TLT VNQ    GLD       QQQ   
Date            ...                                                       
2000-01-03 NaN  ...  NaN  81.050979  92.895111 NaN NaN    NaN  36345200   
2000-01-04 NaN  ...  NaN  77.522399  89.

In [4]:
close_prices = data["Close"]

for ticker in close_prices.columns:
    print(
        ticker,
        close_prices[ticker].first_valid_index(),
        close_prices[ticker].last_valid_index()
    )

GLD 2004-11-18 00:00:00 2025-12-31 00:00:00
QQQ 2000-01-03 00:00:00 2025-12-31 00:00:00
SPY 2000-01-03 00:00:00 2025-12-31 00:00:00
TLT 2002-07-30 00:00:00 2025-12-31 00:00:00
VNQ 2004-09-29 00:00:00 2025-12-31 00:00:00


# 3. Align assets

In [5]:
# We need to align so all 4 ETF:s start at the same time to match future matrix
# For the moment, we work with only one column
close_prices = data["Close"]

aligned_prices = align_assets(close_prices)
print(aligned_prices.head())
print(aligned_prices.shape)
aligned_prices.isna().sum().sum()

Ticker            GLD        QQQ        SPY        TLT        VNQ
Date                                                             
2004-11-18  44.380001  33.120071  79.745285  43.988647  21.189310
2004-11-19  44.779999  32.605846  78.858749  43.637619  20.970301
2004-11-22  44.950001  32.917763  79.234901  43.865047  21.115015
2004-11-23  44.750000  32.867195  79.355743  43.919422  21.204956
2004-11-24  45.049999  33.153790  79.543770  43.919422  21.572582
(5313, 5)


np.int64(0)

# 4. Convert raw to log returns

In [6]:
# Using r_t = log(P_t / P_{t-1})
log_returns = np.log(aligned_prices / aligned_prices.shift(1))

In [7]:
# Remove the first NaN raw
log_returns = log_returns.dropna()
print(log_returns.shape)
log_returns.describe()

(5312, 5)


Ticker,GLD,QQQ,SPY,TLT,VNQ
count,5312.000000,5312.000000,5312.000000,5312.000000,5312.000000
mean,0.000412,0.000549,0.000403,0.000125,0.000265
std,0.011132,0.013615,0.011983,0.009230,0.018122
min,-0.091905,-0.127592,-0.115887,-0.069010,-0.217084
25%,-0.005113,-0.005178,-0.003944,-0.005346,-0.006471
50%,0.000588,0.001146,0.000718,0.000445,0.000768
75%,0.006217,0.007321,0.005776,0.005504,0.007510
max,0.106974,0.114799,0.135578,0.072502,0.157060


# 5. Create rolling observation window

In [8]:
# Seeing only today's return is not relevant enough
# We also care to see how turbulent has the asset been recently
# we compute the std_dev or returns σ_t=std(r_t−19,...,r_t), 20 is approx. 1 month of trades
volatility_20 = log_returns.rolling(20).std()
print(volatility_20.head(25)) # volatility = 0.009785 ==> 0.97%

Ticker           GLD       QQQ       SPY       TLT       VNQ
Date                                                        
2004-11-19       NaN       NaN       NaN       NaN       NaN
2004-11-22       NaN       NaN       NaN       NaN       NaN
2004-11-23       NaN       NaN       NaN       NaN       NaN
2004-11-24       NaN       NaN       NaN       NaN       NaN
2004-11-26       NaN       NaN       NaN       NaN       NaN
2004-11-29       NaN       NaN       NaN       NaN       NaN
2004-11-30       NaN       NaN       NaN       NaN       NaN
2004-12-01       NaN       NaN       NaN       NaN       NaN
2004-12-02       NaN       NaN       NaN       NaN       NaN
2004-12-03       NaN       NaN       NaN       NaN       NaN
2004-12-06       NaN       NaN       NaN       NaN       NaN
2004-12-07       NaN       NaN       NaN       NaN       NaN
2004-12-08       NaN       NaN       NaN       NaN       NaN
2004-12-09       NaN       NaN       NaN       NaN       NaN
2004-12-10       NaN    

In [9]:
# We add momentum to observe the general direction, up or down
# (momentum is supposed to be smaller than volatility, because avg_return << std_dev)
momentum_20 = log_returns.rolling(20).mean()
print(momentum_20.head(25))

Ticker           GLD       QQQ       SPY       TLT       VNQ
Date                                                        
2004-11-19       NaN       NaN       NaN       NaN       NaN
2004-11-22       NaN       NaN       NaN       NaN       NaN
2004-11-23       NaN       NaN       NaN       NaN       NaN
2004-11-24       NaN       NaN       NaN       NaN       NaN
2004-11-26       NaN       NaN       NaN       NaN       NaN
2004-11-29       NaN       NaN       NaN       NaN       NaN
2004-11-30       NaN       NaN       NaN       NaN       NaN
2004-12-01       NaN       NaN       NaN       NaN       NaN
2004-12-02       NaN       NaN       NaN       NaN       NaN
2004-12-03       NaN       NaN       NaN       NaN       NaN
2004-12-06       NaN       NaN       NaN       NaN       NaN
2004-12-07       NaN       NaN       NaN       NaN       NaN
2004-12-08       NaN       NaN       NaN       NaN       NaN
2004-12-09       NaN       NaN       NaN       NaN       NaN
2004-12-10       NaN    

# 6. Generate derived features

In [10]:
# Combine the features
features = pd.concat(
    [
        log_returns,
        volatility_20,
        momentum_20
    ],
    axis=1,
    keys=["ret", "vol", "mom"]
)
# Drop the first 19 NaN
features = features.dropna()

print(features.shape)
print(features.head())

(5293, 15)
                 ret                                               vol  \
Ticker           GLD       QQQ       SPY       TLT       VNQ       GLD   
Date                                                                     
2004-12-17  0.011608 -0.002555 -0.006692 -0.001351  0.011261  0.009785   
2004-12-20  0.003389 -0.006864  0.000251  0.001802 -0.003506  0.009587   
2004-12-21 -0.002710  0.012170  0.007671  0.003032  0.011349  0.009544   
2004-12-22 -0.004533  0.001260  0.002406 -0.003370  0.007428  0.009546   
2004-12-23  0.005663  0.000754  0.000745 -0.003268 -0.007606  0.009506   

                                                         mom            \
Ticker           QQQ       SPY       TLT       VNQ       GLD       QQQ   
Date                                                                     
2004-12-17  0.008715  0.005472  0.007788  0.008974 -0.000215  0.000705   
2004-12-20  0.008043  0.004732  0.007557  0.008570 -0.000494  0.001144   
2004-12-21  0.008209  0.00

# 7. Create Splits

In [11]:
# Train, Validation, Test
train_features = features.loc[:'2018-12-31']
val_features = features.loc['2019-01-01':'2021-12-31']
test_features = features.loc['2022-01-01':]

print("Train:", train_features.shape)
print("Validation:", val_features.shape)
print("Test:", test_features.shape)

print(train_features.index.min(), train_features.index.max())
print(val_features.index.min(), val_features.index.max())
print(test_features.index.min(), test_features.index.max())

Train: (3533, 15)
Validation: (757, 15)
Test: (1003, 15)
2004-12-17 00:00:00 2018-12-31 00:00:00
2019-01-02 00:00:00 2021-12-31 00:00:00
2022-01-03 00:00:00 2025-12-31 00:00:00


# 8. Normalize

In [12]:
# Use z-score standardization
scaler = StandardScaler()
scaler.fit(train_features)

# Transform each split
train_scaled = pd.DataFrame(
    scaler.transform(train_features),
    index=train_features.index,
    columns=train_features.columns,
)

val_scaled = pd.DataFrame(
    scaler.transform(val_features),
    index=val_features.index,
    columns=val_features.columns,
)

test_scaled = pd.DataFrame(
    scaler.transform(test_features),
    index=test_features.index,
    columns=test_features.columns,
)

train_scaled.describe() # mean is approx 0 and std_dev approx 1

ret                                                          \
Ticker           GLD           QQQ           SPY           TLT           VNQ   
count   3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03   
mean    1.206696e-17 -4.022319e-18  9.050219e-18 -1.307254e-17  8.798824e-18   
std     1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00   
min    -7.887357e+00 -7.404397e+00 -8.844751e+00 -6.037207e+00 -1.105059e+01   
25%    -4.762151e-01 -4.158832e-01 -3.478209e-01 -5.826524e-01 -3.491663e-01   
50%     1.779632e-02  4.002309e-02  2.981990e-02  2.900775e-02  2.249141e-02   
75%     5.173026e-01  4.879777e-01  4.319608e-01  5.894288e-01  3.721212e-01   
max     9.127083e+00  8.986759e+00  1.151429e+01  5.822502e+00  7.971935e+00   

                 vol                                                          \
Ticker           GLD           QQQ           SPY           TLT           VNQ   
count   3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03   
mean   -1.287142e-16 -6.435711e-17 -1.608928e-16  1.287142e-16 -3.217856e-17   
std     1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00   
min    -1.569890e+00 -1.311718e+00 -1.089202e+00 -1.794337e+00 -8.087289e-01   
25%    -7.046050e-01 -6.028397e-01 -5.877817e-01 -6.928447e-01 -5.174504e-01   
50%    -2.566559e-01 -2.720537e-01 -2.832281e-01 -2.656780e-01 -3.323132e-01   
75%     4.645889e-01  2.971829e-01  2.448729e-01  5.188359e-01  6.397259e-02   
max     5.882841e+00  6.554035e+00  7.164020e+00  4.757431e+00  6.690882e+00   

                 mom                                                         
Ticker           GLD          QQQ           SPY           TLT           VNQ  
count   3.533000e+03  3533.000000  3.533000e+03  3.533000e+03  3.533000e+03  
mean    3.217856e-17     0.000000 -4.022319e-17  1.206696e-17 -1.005580e-17  
std     1.000142e+00     1.000142  1.000142e+00  1.000142e+00  1.000142e+00  
min    -4.445788e+00    -6.701132 -8.169516e+00 -4.542909e+00 -8.102790e+00  
25%    -5.793645e-01    -0.479189 -4.084945e-01 -6.320353e-01 -4.252625e-01  
50%    -1.929479e-02     0.121015  1.538539e-01  1.085393e-02  1.026405e-01  
75%     6.326394e-01     0.628952  5.590708e-01  5.798376e-01  5.178283e-01  
max     3.992733e+00     4.328696  4.753390e+00  5.881364e+00  6.018171e+00

In [13]:
# Convert time series to window observation
WINDOW_SIZE = 30

train_windows, train_dates = create_windows(
    train_scaled,
    WINDOW_SIZE,
)
val_windows, val_dates = create_windows(
    val_scaled,
    WINDOW_SIZE,
)
test_windows, test_dates = create_windows(
    test_scaled,
    WINDOW_SIZE,
)